#### 0. Update path, import statistics library.

In [ ]:
import sys

caxscripts_path = "/home/arnaldo.filho/Carcara/repo/cax-scripts/caxscripts"
if caxscripts_path not in sys.path:
    sys.path.append(caxscripts_path)

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
from image_statistics import Histogram2DAnalyzer as HiAn

### 1. Get histogram with given parameters and plot it.

##### 1.1 Instantiate Histogram Analyzer.

In [ ]:
# H2DA. Define bins, size, and range for the 2-D histogram.
bins = 200
size = 1_000_000
hist_range = [[-5, 5], [-5, 5]]

sx, sy = 0.5, 1.414  # sx^2 = 0.25, sy^2 = 2.0
sxy = 0.707          # sxy^2  = 0.5
cov = np.array([[sx*sx, sxy*sxy], [sxy*sxy, sy*sy]])

ana = HiAn.from_gaussian(bins=bins, size=size,
                          cov=cov, hist_range=hist_range)
ana

##### 1.2. Add noise to the histogram.

In [ ]:
# The noise level can be adjusted by changing level (60 in this case).
# ana.add_noise(noisetype="gaussian")
ana.add_noise(noisetype="poisson", level=60)

# Raw plot (no ellipses yet).
fig, ax = plt.subplots(figsize=(10, 6))
m = ax.pcolormesh(ana.xedges, ana.yedges, ana.img.T,
                  shading="auto", cmap="viridis")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(ana.xedges[0], ana.xedges[-1])
ax.set_ylim(ana.yedges[0], ana.yedges[-1])
fig.colorbar(m, ax=ax, label="count")
plt.show()


#### 1.3. Plot data without thresholding.

In [ ]:
# Plot data without thresholding.
# The thresholded plot is shown below, to be compared with the fitting.
title = "2-D distribution with sigma ellipses (moments)"
fig, ax = ana.plot(title=title,
                   show_ellipse_axes=True)
plt.show()

#### 2. Analyse histogram and extract sigma-ellipse info.


#### 2.1 Evaluate threshold from entropy analysis.


In [ ]:
entropies, bin_edges, optimal_threshold, nbins = ana.compute_threshold()

fig = ana.plot_entropy(entropies, bin_edges, optimal_threshold)
plt.show()

print(f"Optimal threshold: {optimal_threshold}"
      f"\n Number of bins: {nbins}")


##### 2.2. Statistics purely from moments.

In [ ]:
# Calculate moments and print stats.
img = ana.img_thresholded
hprm_mom_r = ana.compute_moments(img=img)
ana.print_stats(hprm=hprm_mom_r)


##### 2.3. Fitting.
Use statistics from moments as guess.
Plot both results, from moments and fitting.

In [ ]:
# Fit gaussian to the thresholded data and compute moments from the fit.
hprm_fit = ana.fit(hprm=hprm_mom_r, img=ana.img_thresholded)
ana.print_stats(hprm=hprm_fit)

# Plot results from both moments and fitting for comparison.
fig, (axm, axf) = plt.subplots(1, 2, figsize=(15, 6))


title = "2-D distribution with sigma ellipses"
# From moments.
ana.plot(
    hprm=hprm_mom_r,
    fig=fig,
    ax=axm,
    title=title,
    show_ellipse_axes=True,
    center_label="centroid",
    )

# From fitting.
ana.plot(
    hprm=hprm_fit,
    title=title,
    show_ellipse_axes=True,
    center_label="centroid",
    fig=fig,
    ax=axf
    )
plt.show()


### 3. Rotate ellipse, fit and plot it.

##### 3.1. Rotate ellipses.

In [ ]:
# Rotate the histogram to align with the principal axes.
theta   = -np.degrees(ana.hprm_mom['theta'])
rotdata = ndi.rotate(ana.img_thresholded, angle=theta, reshape=False)

# Wrap rotated data in a new analyzer, keeping the same bin edges.
rotana = HiAn(rotdata, ana.xedges, ana.yedges)

##### 3.2. Recalculate statistics for comparison.
Compute moments and fit 2D gaussian.

In [ ]:
# Compute moments for the rotated data.
print('\n###\n### Rotated moments (before fitting):\n###')
hprm_mom_r = rotana.compute_moments()
rotana.print_stats(hprm=hprm_mom_r)

# Compute threshold and moments for the rotated data.
# entropies, bin_edges, optimal_threshold, nbins = rotana.compute_threshold()
# rotana.plot_entropy(entropies, bin_edges, optimal_threshold)
# print(f"###\n### Optimal threshold: {optimal_threshold}"
#       f"\n### Number of bins: {nbins}\n###\n")

# Fit gaussian to the rotated data.
print('\n###\n### Rotated moments after fitting:\n###')
hprm_fit_r = rotana.fit(hprm=hprm_mom_r)
rotana.print_stats(hprm=hprm_fit_r)

#### 3.3. Plot rotated ellipses.

In [ ]:
# Plot results from both moments and fitting for comparison.
fig, (axm, axf) = plt.subplots(1, 2, figsize=(15, 6))

# From moments.
rotana.plot(title=title, show_ellipse_axes=True,
          center_label="centroid",
          hprm=hprm_mom_r,
          fig=fig, ax=axm)

# From fitting.
rotana.plot(title=title, show_ellipse_axes=True,
            center_label="centroid",
            hprm=hprm_fit_r,
            fig=fig, ax=axf)

plt.show()